# s14 - Evasion attack: evaluasi R5 (SURVEY)

**Klaim R5** (`paper-survey.md`): NIDS berbasis ML rapuh terhadap evasion
[F28][F33][F34]. Kami serang model global FL terlatih (dari R3) dengan serangan
yang **sudah ada** di literatur, pada **ruang fitur ter-standardisasi (z-space)**,
dengan perturbasi di-**clip** ke persentil 1-99 tiap fitur. Tidak ada aturan konsistensi
protokol antar-fitur (mis. byte vs paket): hasil = taksiran kerentanan di ruang fitur.

**Serangan diuji:** FGSM, PGD (iteratif), JSMA/saliency (pilih fitur paling
berpengaruh). Semua white-box pada MLP.

**PROTOKOL (dikoreksi).** Model dilatih di EC2 dengan z-score LOKAL per klien, maka tiap
test set klien dinormalisasi dengan statistik train klien itu, dan batas clip diambil dari
train klien itu juga. Hasil dilaporkan per klien (`cic`, `unsw`) dan `union`.

**Definisi ASR (dikoreksi).** `asr` = proporsi serangan yang **terdeteksi sebelum diserang**
tetapi lolos sesudah perturbasi. `raw_miss` = proporsi seluruh serangan yang diprediksi
benign sesudah perturbasi (definisi lama; mencakup serangan yang memang sudah lolos).
`base_recall` dilaporkan agar jelas seberapa sehat model sebelum diserang.

**Prasyarat:** `global_final.npz` dari run R3 ada di S3; partisi s01 ada.
Dilakukan **OFFLINE** (tak perlu EC2). **Output:** `survey_out/r5_evasion.csv`.

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','scikit-learn','boto3'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, io, numpy as np
from sklearn.metrics import matthews_corrcoef
OUTDIR='survey_out'; os.makedirs(OUTDIR,exist_ok=True)
BUCKET='ssh-detection-features-232032302717'; REGION='ap-southeast-1'
BASE='unsw-far/federated_survey'
RUN=os.environ.get('RUN','S_R3')
import boto3; s3=boto3.client('s3',region_name=REGION)
def load_npz_s3(key):
    try: obj=s3.get_object(Bucket=BUCKET,Key=key)
    except Exception as e: print('MISS',key,e); return None
    return np.load(io.BytesIO(obj['Body'].read()),allow_pickle=True)

## 1. Muat model global (R3) + data test; siapkan z-space & batas clip

In [ ]:
gf = load_npz_s3(f'{BASE}/run_{RUN}/global_final.npz')
assert gf is not None, 'PRASYARAT GAGAL: global_final.npz R3 belum ada (jalankan FedAvg EC2 + s03).'
W = [gf[f'w{i}'] for i in range(6)]
P = {}
for nm in ['cic_train', 'cic_test', 'unsw_train', 'unsw_test']:
    sub = 'cic' if nm.startswith('cic') else 'unsw'
    d = load_npz_s3(f'{BASE}/data/{sub}/{nm}.npz')
    P[nm] = (np.asarray(d['X'], float), np.asarray(d['y'], int))
CL = ['cic', 'unsw']
Z = {}   # per klien: data uji di z-space LOKAL (statistik dari TRAIN klien itu = sama dgn training FL)
for c in CL:
    Xtr_c = P[f'{c}_train'][0]
    m = Xtr_c.mean(0); s = Xtr_c.std(0); s[s == 0] = 1
    Ztr = (Xtr_c - m) / s
    Z[c] = dict(X=(P[f'{c}_test'][0] - m) / s, y=P[f'{c}_test'][1],
                lo=np.percentile(Ztr, 1, axis=0), hi=np.percentile(Ztr, 99, axis=0))
    print(c, 'test', Z[c]['X'].shape, 'pos', round(float(Z[c]['y'].mean()), 3))

## 2. Forward + gradien input (white-box)

In [ ]:
relu=lambda x:np.maximum(0,x); sig=lambda x:1/(1+np.exp(-np.clip(x,-30,30)))
def fwd(W,X):
    a1=X@W[0]+W[1]; z1=relu(a1); a2=z1@W[2]+W[3]; z2=relu(a2); a3=z2@W[4]+W[5]
    return sig(a3).ravel(),(a1,z1,a2,z2)
def dX_loss(W,X,y):
    o,(a1,z1,a2,z2)=fwd(W,X); g=(o-y.astype(float))[:,None]
    d2=(g@W[4].T)*(a2>0); d1=(d2@W[2].T)*(a1>0); return d1@W[0].T
def predict(W,X): o,_=fwd(W,X); return (o>=0.5).astype(int)
def mcc(W,X,y): return matthews_corrcoef(y,predict(W,X))

## 3. Serangan (semua CONSTRAINED: clip ke [lo,hi]). Target: flow ATTACK (y=1)

Evasion = buat serangan lolos sebagai benign. Kita ukur hanya pada sampel attack.

In [ ]:
def fgsm(W, X, y, eps, lo, hi):
    return np.clip(X + eps * np.sign(dX_loss(W, X, y)), lo, hi)

def pgd(W, X, y, eps, lo, hi, steps=10, alpha=None):
    alpha = alpha or eps / 4; Xa = X.copy()
    for _ in range(steps):
        Xa = Xa + alpha * np.sign(dX_loss(W, Xa, y))
        Xa = np.clip(np.clip(Xa, X - eps, X + eps), lo, hi)  # dalam bola-eps DAN batas clip
    return Xa

def jsma(W, X, y, eps, lo, hi, k=2):
    # saliency: pilih k fitur dgn |dL/dX| terbesar per-sampel, perturbasi hanya itu
    d = dX_loss(W, X, y); Xa = X.copy()
    idxk = np.argsort(-np.abs(d), axis=1)[:, :k]
    for i in range(len(X)):
        j = idxk[i]; Xa[i, j] = np.clip(X[i, j] + eps * np.sign(d[i, j]), lo[j], hi[j])
    return Xa

for c in CL:
    y = Z[c]['y']; X = Z[c]['X']
    print(c, 'baseline recall pada attack (clean):', round(float((predict(W, X)[y == 1] == 1).mean()), 3))

## 4. Sweep epsilon untuk tiap serangan -> attack success rate + ΔMCC

In [ ]:
def run_attack(fn, eps, **kw):
    out = {}; ys = []; p0s = []; p1s = []; dets = []
    for c in CL:
        X, y, lo, hi = Z[c]['X'], Z[c]['y'], Z[c]['lo'], Z[c]['hi']
        atk = (y == 1)
        p0 = predict(W, X)
        Xa = X.copy(); Xa[atk] = fn(W, X[atk], y[atk], eps, lo, hi, **kw)
        p1 = predict(W, Xa)
        det = atk & (p0 == 1)                      # serangan yang TERDETEKSI sebelum diserang
        out[c] = (y, p0, p1, det)
        ys.append(y); p0s.append(p0); p1s.append(p1); dets.append(det)
    out['union'] = (np.concatenate(ys), np.concatenate(p0s), np.concatenate(p1s), np.concatenate(dets))
    return out

def summarize(y, p0, p1, det):
    atk = (y == 1)
    return dict(base_recall=float((p0[atk] == 1).mean()),
                raw_miss=float((p1[atk] == 0).mean()),
                asr=float((p1[det] == 0).mean()) if det.sum() > 0 else float('nan'),
                mcc_clean=float(matthews_corrcoef(y, p0)),
                mcc_under=float(matthews_corrcoef(y, p1)))

rows = []
for eps in [0.05, 0.1, 0.2, 0.5]:
    for nm, fn in [('FGSM', fgsm), ('PGD', pgd), ('JSMA', jsma)]:
        res = run_attack(fn, eps)
        for scope, (y, p0, p1, det) in res.items():
            r = summarize(y, p0, p1, det)
            rows.append(dict(scope=scope, attack=nm, eps=eps, **r))
        r = rows[-1]   # union
        print(f'{nm:5s} eps={eps:<4} [union] base_recall={r["base_recall"]:.3f} ASR(detected->evade)={r["asr"]:.3f} '
              f'raw_miss={r["raw_miss"]:.3f} MCC {r["mcc_clean"]:+.3f} -> {r["mcc_under"]:+.3f}')

## 5. Simpan R5 -> CSV + S3

In [ ]:
import csv
path = os.path.join(OUTDIR, 'r5_evasion.csv')
fields = ['scope', 'attack', 'eps', 'base_recall', 'raw_miss', 'asr', 'mcc_clean', 'mcc_under']
with open(path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fields); w.writeheader()
    for r in rows: w.writerow(r)
print('saved', path)
try:
    s3.upload_file(path, BUCKET, f'{BASE}/results/r5_evasion.csv'); print('upload OK')
except Exception as e:
    print('(upload dilewati:', e, ')')
print('=== s14 SELESAI. asr = serangan terdeteksi-lalu-lolos; bandingkan dgn raw_miss & base_recall. ===')